# Using DFTpy mixer

This notebook demonstrates four different ways to drive SCF density/potential mixing with a QEpy iterative `Driver`, using the same bulk-aluminum test system throughout so the converged total energy (≈ -137.9145 Ry) can be compared across approaches:

1. **QE's built-in mixer** — the default, via `driver.mix()`.
2. **Manual linear density mixing** — blend the density in Python using DFTpy, bypassing QE's mixer entirely.
3. **Manual linear potential mixing** — blend the Hartree+XC potential instead of the density.
4. **DFTpy's Pulay mixer** — a drop-in, faster-converging replacement for the manual linear mixer in (2).

It finishes by exporting the converged density to an XSF file for visualization.</cell id="5d0a114b">


In [ ]:
# --- optional pip installs (normally leave collapsed / do not run) ---
!pip install qepy f90wrap==0.2.16
!pip install dftpy
!pip install matplotlib

## Imports

QEpy's `Driver`/`QEInput` plus NumPy. (The ASE/matplotlib imports below are unused in this notebook — left over from a template.)

In [1]:
import qepy
import importlib

In [2]:
import numpy as np
from qepy.driver import Driver
from qepy.io import QEInput

In [3]:
from ase.io.trajectory import Trajectory
from ase.lattice.hexagonal import Graphene
from ase import Atoms
import matplotlib.pyplot as plt

## System: bulk aluminum

Same FCC aluminum test cell used elsewhere in this folder (1 atom/cell, ONCV PBE pseudopotential, smeared occupations, `ecutwfc=30` Ry, 4×4×4 k-mesh) — small enough to iterate on quickly, with a tight `conv_thr=1e-8` so the different mixers can be compared precisely.

In [4]:
qe_options = {
    '&control': {
        'calculation': "'scf'",
        'pseudo_dir': "'./data/'"
    },
    '&system': {
        'ibrav' : 0,
        'degauss': 0.005,
        'ecutwfc': 30,
        'nat': 1,
        'ntyp': 1,
        'occupations': "'smearing'"
    },
    '&electrons': {
        'conv_thr' : 1e-8
    },
    'atomic_positions crystal': ['Al    0.0  0.0  0.0'],
    'atomic_species': ['Al  26.98 Al_ONCV_PBE-1.2.upf'],
    'k_points automatic': ['4 4 4 0 0 0'],
    'cell_parameters angstrom':[
        '0.     2.025  2.025',
        '2.025  0.     2.025',
        '2.025  2.025  0.   '],
}

## 1) Baseline: QE's built-in density mixer

Call `driver.mix()` after each `diagonalize()` — this uses QE's own (Broyden-type) mixer internally. Loop until `check_convergence()` is satisfied, then `calc_energy()` gives the reference total energy that the other three approaches below should reproduce.

In [5]:
driver=Driver(qe_options=qe_options, iterative = True, logfile='tmp.out')

In [6]:
for i in range(60):
    driver.diagonalize()
    driver.mix()
    converged = driver.check_convergence()
    print ('Iter: ',i,' - Conv: ', driver.get_scf_error())
    if converged : break
driver.calc_energy()

Iter:  0  - Conv:  0.07520824059877537
Iter:  1  - Conv:  0.001394918464705383
Iter:  2  - Conv:  3.710927074023805e-05
Iter:  3  - Conv:  1.3298945184849757e-07
Iter:  4  - Conv:  4.2953143005617965e-10


-137.91449181100822

## 2) Manual linear density mixing via DFTpy

Bypass QE's mixer entirely: after each `diagonalize()`, linearly blend the old and new densities in Python (`rho = (1-coef)*rho + coef*rho_new`, `coef=0.7`), then push the mixed density back into QE with `set_density()`. DFTpy's `Hartree.compute` on the density *difference* gives a convenient scalar residual to test convergence against (in place of QE's internal SCF-error check), and `nc` tracks the corresponding change in electron count. Call `end_scf()` once converged to finalize.

In [7]:
from dftpy.functional import Hartree

In [8]:
driver=Driver(qe_options=qe_options, iterative = True, logfile='tmp.1.out')

In [9]:
rho = driver.get_density().copy()
coef = 0.7
for i in range(20):
    driver.diagonalize()
    #
    rho_new = driver.get_density().copy()
    drho = driver.data2field(rho_new-rho)
    error = Hartree.compute(drho).energy
    nc = np.abs(drho).sum()*driver.get_volume()/drho.size
    print ('Iter: ',i,' - Conv: ', error, 'dN:', nc)
    if error < 1e-8:
        driver.end_scf()
        break
    #
    rho = (1-coef)*rho + coef * rho_new
    driver.set_density(rho)
driver.calc_energy()

Iter:  0  - Conv:  0.03752851919933115 dN: 0.6417690036360973
Iter:  1  - Conv:  0.0007030693684957934 dN: 0.15211482969792478
Iter:  2  - Conv:  3.8992346521276593e-05 dN: 0.04568664571106663
Iter:  3  - Conv:  3.35928338770151e-06 dN: 0.014066196101718872
Iter:  4  - Conv:  3.233322371489466e-07 dN: 0.0044164521276377575
Iter:  5  - Conv:  3.051837084356297e-08 dN: 0.0013627641258597595
Iter:  6  - Conv:  2.804704165009501e-09 dN: 0.0004196697294594821


-137.91449181103968

## 3) Manual linear mixing of the Hartree+XC potential

Instead of mixing the density, mix the *potential*: `get_hartree()` and `get_exchange_correlation()` each return `(potential, energy)`; sum their potentials into `v_hxc` and linearly blend it across iterations the same way as before. The mixed potential (plus its energy) is injected via `set_external_potential(v_hxc, exttype=('hartree', 'xc'), extene=extene)`, so QE still handles the remaining potential terms (e.g. the local pseudopotential) internally.

In [10]:
driver=Driver(qe_options=qe_options, iterative = True, logfile='tmp.2.out')

In [11]:
rho_new = driver.get_density().copy()
coef = 0.7
hartree = driver.get_hartree()
xc = driver.get_exchange_correlation()
v_hxc = hartree[0] + xc[0]
for i in range(10):
    driver.diagonalize()
    #
    rho_new, rho = driver.get_density().copy(), rho_new
    drho = driver.data2field(rho_new-rho)
    error = Hartree.compute(drho).energy
    nc = np.abs(drho).sum()*driver.get_volume()/drho.size
    print ('Iter: ',i,' - Conv: ', error, 'dN:', nc)
    if error < 1e-8:
        driver.end_scf()
        break
    #
    hartree = driver.get_hartree()
    xc = driver.get_exchange_correlation()
    v_hxc_new = hartree[0] + xc[0]
    v_hxc = (1-coef)*v_hxc + coef*v_hxc_new
    extene = hartree[1] + xc[1]
    driver.set_external_potential(v_hxc, exttype=('hartree', 'xc'), extene=extene)
driver.get_energy()

Iter:  0  - Conv:  0.0374379019155871 dN: 0.6409983057290252
Iter:  1  - Conv:  0.0013992599106396775 dN: 0.11937626385346811
Iter:  2  - Conv:  1.2556386014546973e-05 dN: 0.011465237457737738
Iter:  3  - Conv:  3.3180164243438324e-07 dN: 0.002069687812913146
Iter:  4  - Conv:  7.840132496965606e-09 dN: 0.000440317618982


-137.9144915709664

## 4) DFTpy's Pulay mixer

Swap the naive linear-mixing loop in (2) for DFTpy's built-in `Mixer` (Pulay/DIIS-style), which generally converges faster and more robustly. Density values are converted to/from DFTpy `field` objects (`data2field`/`field2data`) so the mixer can operate on them; note `drho.integral()` replaces the manual `np.abs(...).sum() * volume / size` used earlier for the same "change in electron count" metric.

In [12]:
driver=Driver(qe_options=qe_options, iterative = True, logfile='tmp.3.out')

In [13]:
from dftpy.mixer import Mixer
driver.mixer = Mixer(scheme='pulay')

In [14]:
rho = driver.data2field(driver.get_density().copy())
for i in range(10):
    driver.diagonalize()
    #
    rho_new = driver.data2field(driver.get_density().copy())
    drho = rho_new - rho
    error = Hartree.compute(drho).energy
    nc = np.abs(drho).integral()
    print ('Iter: ',i,' - Conv: ', error, 'dN:', nc)
    if error < 1e-8:
        driver.end_scf()
        break
    #
    rho = driver.mixer(rho, rho_new, coef=0.7)
    density = driver.field2data(rho)
    driver.set_density(density)
driver.calc_energy()

Iter:  0  - Conv:  0.037554510773110436 dN: 0.6418754781189862
Iter:  1  - Conv:  0.0007007576828602049 dN: 0.15178804133324647
Iter:  2  - Conv:  1.3737606368093629e-05 dN: 0.028148121017133258
Iter:  3  - Conv:  8.152458946043758e-08 dN: 0.0013629194467525499
Iter:  4  - Conv:  1.809314284809117e-10 dN: 0.00012862004000021847


-137.91449180811244

## 5) Export the converged density

Get the ion structure with `get_dftpy_ions()`, then write the real-space density field to an XSF file for visualization in tools like VESTA or XCrySDen.

In [15]:
ions = driver.get_dftpy_ions()

In [16]:
rho.write('tmp.xsf', ions=ions)